In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("News Data Analysis") \
    .getOrCreate()

# Define the schema manually
schema = StructType([
    StructField("timestamp", IntegerType(), True),
    StructField("source", StringType(), True),
    StructField("archive", StringType(), True),
    StructField("id", IntegerType(), True),
    StructField("probability", FloatType(), True),
    StructField("keywords", MapType(StringType(), IntegerType()), True),
    StructField("sentiment", FloatType(), True),
    #StructField("status", StringType(), True),
    #StructField("error", StringType(), True)
])

# Load the DataFrame with the defined schema
df = spark.read.format("json").schema(schema).load("data/news/status=success")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/12/09 21:45:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
length = df.count()
files_loaded = df.inputFiles()
print(f"Number of rows in the DataFrame: {length}")
print(f"Files loaded by Spark: {len(files_loaded)}")

df.printSchema()

Number of rows in the DataFrame: 8991
Files loaded by Spark: 311
root
 |-- timestamp: integer (nullable = true)
 |-- source: string (nullable = true)
 |-- archive: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- probability: float (nullable = true)
 |-- keywords: map (nullable = true)
 |    |-- key: string
 |    |-- value: integer (valueContainsNull = true)
 |-- sentiment: float (nullable = true)



In [3]:
df.show()

+---------+---------+--------------------+------+-----------+--------------------+-----------+
|timestamp|   source|             archive|    id|probability|            keywords|  sentiment|
+---------+---------+--------------------+------+-----------+--------------------+-----------+
|   201005|O Mirante|https://arquivo.p...|130244|  0.5438319|{função -> 2, aut...|-0.49929795|
|   201005|O Mirante|https://arquivo.p...|130381|  0.5400149|{casa -> 2, ofert...| 0.65770584|
|   201005|O Mirante|https://arquivo.p...|130370| 0.51225215|{função -> 2, cas...|   -0.50018|
|   201005|O Mirante|https://arquivo.p...|130485|  0.5800057|{reduzir -> 2, Tr...| -0.5516859|
|   201005|O Mirante|https://arquivo.p...|130170| 0.51693547|{GPL -> 1, aula -...| 0.63821393|
|   201005|O Mirante|https://arquivo.p...|130228| 0.54250735|{autarquia -> 2, ...|  0.4899384|
|   201005|O Mirante|https://arquivo.p...|130178|  0.5090795|{função -> 2, alv...| -0.5223741|
|   201005|O Mirante|https://arquivo.p...|130257| 

In [4]:
from pyspark.sql import functions as F

keyword = "saúde"

# Check if the key "X" exists in the "keywords" map
#df_with_key = df.filter(df["keywords"].getItem(keyword).isNotNull())
df_with_key = df.filter(F.col("keywords").getItem(keyword).isNotNull() & (F.col("keywords").getItem(keyword) > 2))

length = df_with_key.count()
print(f"Number of rows in the DataFrame: {length}")

df_with_key.show(truncate=False)

Number of rows in the DataFrame: 147
+---------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+-----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [5]:
# Extract and sum all key-value pairs
result = (
    df_with_key.rdd
    .flatMap(lambda row: row["keywords"].items())  # Extract key-value pairs
    .reduceByKey(lambda x, y: x + y)             # Sum values by key
    .collect()                                   # Collect result
)

# Convert to dictionary
final_dict = dict(result)

final_dict = {k: {"count": v} for k, v in final_dict.items()}
sorted_dict = dict(sorted(final_dict.items(), key=lambda item: item[1]["count"], reverse=True))
sorted_dict

{'saúde': {'count': 726},
 'ano': {'count': 376},
 'criança': {'count': 288},
 'pessoa': {'count': 260},
 'droga': {'count': 228},
 'ter': {'count': 220},
 'direito': {'count': 182},
 'falar': {'count': 160},
 'Sociedade': {'count': 152},
 'Santarém': {'count': 152},
 'problema': {'count': 148},
 'concelho': {'count': 144},
 'situação': {'count': 142},
 'caso': {'count': 142},
 'passar': {'count': 140},
 'pensar': {'count': 138},
 'casa': {'count': 132},
 'família': {'count': 132},
 'ficar': {'count': 130},
 'Mirante': {'count': 130},
 'fazer': {'count': 128},
 'presidente': {'count': 126},
 'dizer': {'count': 124},
 'unidade': {'count': 124},
 'banho': {'count': 124},
 'cidade': {'count': 116},
 'andar': {'count': 112},
 'médico': {'count': 112},
 'tomar': {'count': 112},
 'interesse': {'count': 110},
 'CRIL': {'count': 108},
 'deixar': {'count': 108},
 'processo': {'count': 108},
 'serviço': {'count': 106},
 'hospital': {'count': 106},
 'população': {'count': 106},
 'escola': {'count